<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/research%2Fpost-e50-axial-level-pairing/68_postE50_lumbar_morphometric_measurements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 68 · Post-E50 — Lumbar Morphometric Measurements

**Roadmap:** 67A → 67B/67B1/67B2 → 67C → **68 morphometric measurements**.

## Índice
- **68-A — Contrato, identidad de artefactos y scope**
- **68-B — Mediciones sagitales**
- **68-C — Mediciones axiales**
- **68-D — QC físico, estabilidad y auditoría**
- **68-E — Contrato final para Backend/Frontend**

### Contrato
- No entrena ni modifica modelos.
- Desarrollo técnico sobre **outer TRAIN** únicamente.
- `internal_test` y test oficial permanecen bloqueados.
- 67C permanece cerrado como **NOT VALIDATED**.
- Las mediciones son geométricas/revisables, no diagnósticas.
- Las clases axiales se conservan como `raw_*`: el repositorio no tiene nombres clínicos corroborados para ellas.
- **Runtime recomendado: NVIDIA L4.** 68-A/D/E son CPU-friendly; 68-B/C usan inferencia y conviene L4.


# 68-A — Contrato, identidad de artefactos y scope

In [2]:

import hashlib, json, math, os, random, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
from typing import Mapping
import numpy as np
import pandas as pd

SEED=2026
random.seed(SEED); np.random.seed(SEED)

try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

PFI_DRIVE_ROOT=Path(os.environ.get("PFI_DRIVE_ROOT","/content/drive/MyDrive/PFI_MVP" if IN_COLAB else "./PFI_MVP"))
RSNA_ROOT=PFI_DRIVE_ROOT/"data"/"RSNA_LUMBAR_DISC"
TRAIN_IMAGES_ROOT=RSNA_ROOT/"train_images"
RESULTS_DIR=PFI_DRIVE_ROOT/"results"/"post_e50"/"68"
METRICS_DIR=PFI_DRIVE_ROOT/"metrics"/"post_e50"/"68"
for p in (RESULTS_DIR,METRICS_DIR): p.mkdir(parents=True,exist_ok=True)

SAGITTAL_CHECKPOINT_PATH=PFI_DRIVE_ROOT/"models"/"final"/"sagittal_spider_multiclass_final_best_cf11dcc0.pt"
AXIAL_CHECKPOINT_PATH=PFI_DRIVE_ROOT/"models"/"final"/"axial_t2_alkafri_final_v2_candidate.pt"

EXPECTED_SAGITTAL_SHA256="cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944"
EXPECTED_AXIAL_SHA256 = ("a48cbddd858b5615010fd809412f3d17dae6871fbe12a38f4720e6f6bc70f739")
EXPECTED_CSV_SHA256={
 "train.csv":"f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528",
 "train_label_coordinates.csv":"416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635",
 "train_series_descriptions.csv":"bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33",
}

RSNA_INTERNAL_TEST_LOCKED=True
OFFICIAL_TEST_ACCESSED=False
TRAINING_PERFORMED=False
POST_VALIDATION_RETUNING_67C=False

# Primera corrida: 30. Si todo pasa, cambiar SOLO a None.
MAX_STUDIES_68=None
SAVE_MASKS=False

assert RSNA_INTERNAL_TEST_LOCKED and not OFFICIAL_TEST_ACCESSED and not TRAINING_PERFORMED and not POST_VALIDATION_RETUNING_67C
print("IN_COLAB:",IN_COLAB)
print("PFI_DRIVE_ROOT:",PFI_DRIVE_ROOT)
print("MAX_STUDIES_68:",MAX_STUDIES_68)


Mounted at /content/drive
IN_COLAB: True
PFI_DRIVE_ROOT: /content/drive/MyDrive/PFI_MVP
MAX_STUDIES_68: None


In [3]:

def _importable(name):
    try:
        m=__import__(name); return getattr(m,"__version__","unknown")
    except ImportError: return None

req={k:_importable(k) for k in ("torch","pydicom","scipy","PIL")}
pkgs=[]
if req["pydicom"] is None: pkgs.append("pydicom")
if req["scipy"] is None: pkgs.append("scipy")
if req["PIL"] is None: pkgs.append("Pillow")
if pkgs: subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*pkgs])

import torch, torch.nn as nn, pydicom, scipy
from PIL import Image
from scipy import ndimage

torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:",DEVICE)
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))
else: print("WARNING: para 68-B/68-C usar L4.")


DEVICE: cuda
GPU: NVIDIA L4


In [4]:

CANONICAL_LEVELS=["L1-L2","L2-L3","L3-L4","L4-L5","L5-S1"]
SAGITTAL_CLASS_NAMES={0:"background",1:"vertebra_group",2:"canal",3:"disc_group"}
AXIAL_CLASS_NAMES={0:"background_250",1:"raw_0",2:"raw_50",3:"raw_100",4:"raw_150",5:"raw_200"}

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

def opaque_id(x): return hashlib.sha256(str(x).encode()).hexdigest()[:12]

def build_split(ids,seed=2026):
    ordered=sorted(str(x) for x in ids); rng=np.random.default_rng(seed); sh=list(ordered); rng.shuffle(sh)
    n=len(sh); a=int(n*.70); b=int(n*.15)
    return {"train":sorted(sh[:a]),"validation":sorted(sh[a:a+b]),"internal_test":sorted(sh[a+b:])}

for fn,exp in EXPECTED_CSV_SHA256.items():
    p=RSNA_ROOT/fn; assert p.is_file(),p
    got=sha256_file(p); print(fn,got,"MATCH" if got==exp else "MISMATCH"); assert got==exp

series_df=pd.read_csv(RSNA_ROOT/"train_series_descriptions.csv",dtype={"study_id":"string","series_id":"string"})
outer_split=build_split(series_df.study_id.dropna().unique())
assert {k:len(v) for k,v in outer_split.items()}=={"train":1382,"validation":296,"internal_test":297}

TRAIN_STUDY_IDS=outer_split["train"]
RUN_STUDY_IDS=TRAIN_STUDY_IDS if MAX_STUDIES_68 is None else TRAIN_STUDY_IDS[:MAX_STUDIES_68]
print("Outer split:",{k:len(v) for k,v in outer_split.items()})
print("68 scope: OUTER TRAIN ONLY | run:",len(RUN_STUDY_IDS))


train.csv f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528 MATCH
train_label_coordinates.csv 416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635 MATCH
train_series_descriptions.csv bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33 MATCH
Outer split: {'train': 1382, 'validation': 296, 'internal_test': 297}
68 scope: OUTER TRAIN ONLY | run: 1382


In [5]:
from pathlib import Path

search_roots = [
    PFI_DRIVE_ROOT / "models",
    PFI_DRIVE_ROOT / "results",
    PFI_DRIVE_ROOT / "checkpoints",
]

candidates = []

for root in search_roots:
    if not root.exists():
        continue

    print("\nSearching:", root)

    for pattern in (
        "*axial*.pt",
        "*axial*.pth",
        "*alkafri*.pt",
        "*alkafri*.pth",
        "*v2*.pt",
        "*v2*.pth",
    ):
        found = list(root.rglob(pattern))
        print(pattern, "->", len(found))
        candidates.extend(found)

# remove duplicates
candidates = sorted(set(candidates))

print("\nTOTAL CANDIDATES:", len(candidates))

for p in candidates:
    print(p)


Searching: /content/drive/MyDrive/PFI_MVP/models
*axial*.pt -> 4
*axial*.pth -> 0
*alkafri*.pt -> 0
*alkafri*.pth -> 0
*v2*.pt -> 0
*v2*.pth -> 0

TOTAL CANDIDATES: 4
/content/drive/MyDrive/PFI_MVP/models/E10_axial_t2_final_training_clean_best.pt
/content/drive/MyDrive/PFI_MVP/models/E9_axial_t2_final_labels_unet_best_state_dict.pt
/content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_best.pt
/content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_v2_candidate.pt


In [6]:
EXPECTED = EXPECTED_AXIAL_SHA256

print("Expected SHA:", EXPECTED)
print()

matches = []

for i, p in enumerate(candidates, start=1):
    try:
        got = sha256_file(p)

        print(
            f"[{i}/{len(candidates)}]",
            p.name,
            got
        )

        if got == EXPECTED:
            matches.append(p)

    except Exception as e:
        print("ERROR:", p, type(e).__name__, e)

print("\nMATCHES:")

for p in matches:
    print(p)

assert len(matches) >= 1, (
    "No axial checkpoint matched the expected SHA."
)

AXIAL_CHECKPOINT_PATH = matches[0]

print("\nUsing:")
print(AXIAL_CHECKPOINT_PATH)

Expected SHA: a48cbddd858b5615010fd809412f3d17dae6871fbe12a38f4720e6f6bc70f739

ERROR: /content/drive/MyDrive/PFI_MVP/models/E10_axial_t2_final_training_clean_best.pt FileNotFoundError [Errno 2] No such file or directory: '/content/drive/MyDrive/PFI_MVP/models/E10_axial_t2_final_training_clean_best.pt'
ERROR: /content/drive/MyDrive/PFI_MVP/models/E9_axial_t2_final_labels_unet_best_state_dict.pt FileNotFoundError [Errno 2] No such file or directory: '/content/drive/MyDrive/PFI_MVP/models/E9_axial_t2_final_labels_unet_best_state_dict.pt'
ERROR: /content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_best.pt FileNotFoundError [Errno 2] No such file or directory: '/content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_best.pt'
ERROR: /content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_v2_candidate.pt FileNotFoundError [Errno 2] No such file or directory: '/content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_v2_candidate.pt'

MATCHES:


AssertionError: No axial checkpoint matched the expected SHA.

## Runtime compartido — arquitecturas exactas + morfometría física

In [ ]:

class SDC(nn.Module):
    def __init__(self,i,o):
        super().__init__(); self.block=nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(self,x): return self.block(x)

class SagUNet(nn.Module):
    def __init__(self,num_classes=4,base_channels=16):
        super().__init__(); b=base_channels
        self.enc1=SDC(1,b); self.pool1=nn.MaxPool2d(2); self.enc2=SDC(b,b*2); self.pool2=nn.MaxPool2d(2); self.enc3=SDC(b*2,b*4); self.pool3=nn.MaxPool2d(2)
        self.bottleneck=SDC(b*4,b*8); self.up3=nn.ConvTranspose2d(b*8,b*4,2,2); self.dec3=SDC(b*8,b*4)
        self.up2=nn.ConvTranspose2d(b*4,b*2,2,2); self.dec2=SDC(b*4,b*2); self.up1=nn.ConvTranspose2d(b*2,b,2,2); self.dec1=SDC(b*2,b); self.out_conv=nn.Conv2d(b,num_classes,1)
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool1(e1)); e3=self.enc3(self.pool2(e2)); z=self.bottleneck(self.pool3(e3))
        d3=self.dec3(torch.cat([self.up3(z),e3],1)); d2=self.dec2(torch.cat([self.up2(d3),e2],1)); d1=self.dec1(torch.cat([self.up1(d2),e1],1)); return self.out_conv(d1)

class ADC(nn.Module):
    def __init__(self,i,o):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(self,x): return self.net(x)

class AxUNet(nn.Module):
    def __init__(self,num_classes=6,base_channels=16):
        super().__init__(); b=base_channels
        self.e1=ADC(1,b); self.e2=ADC(b,b*2); self.e3=ADC(b*2,b*4); self.pool=nn.MaxPool2d(2); self.mid=ADC(b*4,b*8)
        self.u3=nn.ConvTranspose2d(b*8,b*4,2,2); self.d3=ADC(b*8,b*4); self.u2=nn.ConvTranspose2d(b*4,b*2,2,2); self.d2=ADC(b*4,b*2); self.u1=nn.ConvTranspose2d(b*2,b,2,2); self.d1=ADC(b*2,b); self.out=nn.Conv2d(b,num_classes,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); z=self.mid(self.pool(e3))
        x=self.d3(torch.cat([self.u3(z),e3],1)); x=self.d2(torch.cat([self.u2(x),e2],1)); x=self.d1(torch.cat([self.u1(x),e1],1)); return self.out(x)

def _state(ck):
    if isinstance(ck,Mapping):
        for k in ("model_state_dict","state_dict","model"):
            if isinstance(ck.get(k),Mapping): sd=ck[k]; break
        else: sd=ck
    else: raise ValueError("checkpoint")
    out={}
    for k,v in sd.items():
        kk=str(k)
        for p in ("module.","model."):
            if kk.startswith(p): kk=kk[len(p):]
        out[kk]=v
    return out

def load_model(path,kind):
    ck=torch.load(path,map_location=DEVICE,weights_only=False); sd=_state(ck)
    if kind=="sag":
        b=int(ck.get("base_channels",sd["enc1.block.0.weight"].shape[0])); n=int(ck.get("num_classes",sd["out_conv.weight"].shape[0])); m=SagUNet(n,b)
    else:
        b=int(ck.get("base_channels",sd["e1.net.0.weight"].shape[0])); n=int(ck.get("num_classes",sd["out.weight"].shape[0])); m=AxUNet(n,b)
    m.load_state_dict(sd,strict=True); m.to(DEVICE).eval()
    target=tuple(int(x) for x in ck.get("target_size",(256,256)))
    return m,target

sag_model,SAG_TARGET=load_model(SAGITTAL_CHECKPOINT_PATH,"sag")
ax_model,AX_TARGET=load_model(AXIAL_CHECKPOINT_PATH,"ax")
print("Models loaded:",SAG_TARGET,AX_TARGET)


In [ ]:

def robust_norm(a):
    x=np.asarray(a,dtype=np.float32); finite=np.isfinite(x)
    if not finite.any(): return np.zeros_like(x)
    lo,hi=np.percentile(x[finite],[1,99])
    if hi<=lo: return np.zeros_like(x)
    return np.clip((x-lo)/(hi-lo+1e-8),0,1).astype(np.float32)

def infer_mask(model,arr,target):
    x=robust_norm(arr); im=Image.fromarray((x*255).astype(np.uint8)).resize((target[1],target[0]),Image.Resampling.BILINEAR)
    t=torch.from_numpy(np.asarray(im,dtype=np.float32)[None,None]/255.0).to(DEVICE)
    with torch.inference_mode(): p=model(t).argmax(1)[0].cpu().numpy().astype(np.uint8)
    native=Image.fromarray(p).resize((arr.shape[1],arr.shape[0]),Image.Resampling.NEAREST)
    return np.asarray(native,dtype=np.uint8)

def read_dicom(p):
    ds=pydicom.dcmread(str(p)); arr=ds.pixel_array.astype(np.float32)
    spacing=tuple(float(v) for v in ds.PixelSpacing) if hasattr(ds,"PixelSpacing") else None
    return ds,arr,spacing

def ordered_paths(study,series):
    root=TRAIN_IMAGES_ROOT/str(study)/str(series); rec=[]
    for p in root.glob("*.dcm"):
        try:
            ds=pydicom.dcmread(str(p),stop_before_pixels=True,specific_tags=["InstanceNumber"])
            rec.append((int(getattr(ds,"InstanceNumber",0)),p))
        except Exception: pass
    return [p for _,p in sorted(rec)]

def comps(binary,min_pixels=20):
    lab,n=ndimage.label(binary.astype(np.uint8)); out=[]
    for i in range(1,n+1):
        m=lab==i
        if int(m.sum())>=min_pixels: out.append(m)
    return sorted(out,key=lambda m:int(m.sum()),reverse=True)

def morph(mask,spacing):
    if spacing is None or not mask.any(): return None
    rs,cs=map(float,spacing); rr,cc=np.where(mask); pts=np.column_stack([rr*rs,cc*cs]).astype(float); pts-=pts.mean(0)
    eig=np.linalg.eigh(np.cov(pts,rowvar=False))[1] if len(pts)>=3 else np.eye(2); pr=pts@eig
    lengths=np.ptp(pr,axis=0) if len(pts)>=3 else np.array([0.,0.]); major,minor=sorted(map(float,lengths),reverse=True)
    return {
      "area_mm2":float(mask.sum())*rs*cs,
      "bbox_height_mm":float(rr.max()-rr.min()+1)*rs,
      "bbox_width_mm":float(cc.max()-cc.min()+1)*cs,
      "major_axis_mm":major,"minor_axis_mm":minor,
      "centroid_row_px":float(rr.mean()),"centroid_col_px":float(cc.mean()),"pixel_count":int(mask.sum())
    }

m=np.zeros((10,20),bool); m[2:7,5:15]=True; q=morph(m,(2.,.5))
assert abs(q["area_mm2"]-50)<1e-9 and abs(q["bbox_height_mm"]-10)<1e-9 and abs(q["bbox_width_mm"]-5)<1e-9
print("Morphometry physical-unit self-test: PASS")


In [ ]:
# ============================================================
# 68-B — SALVAGE CURRENT IN-MEMORY PROGRESS
#
# Ejecutar UNA VEZ inmediatamente después de interrumpir
# la celda 68-B original.
# ============================================================

import json
from pathlib import Path
import pandas as pd

B_PARTIAL_PATH = RESULTS_DIR / "68_B_sagittal_measurements.partial.csv"
B_FAIL_PARTIAL_PATH = RESULTS_DIR / "68_B_sagittal_failures.partial.csv"
B_DONE_PATH = RESULTS_DIR / "68_B_done_studies.json"

# Recupera lo que ya quedó en memoria.
current_rows = list(rows) if "rows" in globals() else []
current_failures = list(failures) if "failures" in globals() else []

partial_df = pd.DataFrame(current_rows)
partial_fail_df = pd.DataFrame(current_failures)

# Identificamos estudios realmente registrados.
done_opaque = set()

if not partial_df.empty and "study_id_opaque" in partial_df.columns:
    done_opaque.update(
        partial_df["study_id_opaque"]
        .dropna()
        .astype(str)
        .tolist()
    )

if not partial_fail_df.empty and "study_id_opaque" in partial_fail_df.columns:
    done_opaque.update(
        partial_fail_df["study_id_opaque"]
        .dropna()
        .astype(str)
        .tolist()
    )

# Si el loop fue interrumpido mientras estaba dentro de un estudio,
# NO marcamos automáticamente ese estudio como terminado.
# Si quedaron algunas filas parciales, el resumable las eliminará
# antes de reprocesarlo.

partial_df.to_csv(
    B_PARTIAL_PATH,
    index=False,
)

partial_fail_df.to_csv(
    B_FAIL_PARTIAL_PATH,
    index=False,
)

B_DONE_PATH.write_text(
    json.dumps(
        sorted(done_opaque),
        indent=2,
    ),
    encoding="utf-8",
)

print("=" * 65)
print("68-B CURRENT PROGRESS SALVAGED")
print("=" * 65)

print("Measurement rows:", len(partial_df))
print("Failure rows:", len(partial_fail_df))
print("Studies definitely completed:", len(done_opaque))

print()
print("Partial measurements:", B_PARTIAL_PATH)
print("Partial failures:", B_FAIL_PARTIAL_PATH)
print("Done manifest:", B_DONE_PATH)

if B_PARTIAL_PATH.is_file():
    print("Partial SHA:", sha256_file(B_PARTIAL_PATH))

print()
print("SAFE TO START RESUMABLE 68-B")

68-B CURRENT PROGRESS SALVAGED
Measurement rows: 0
Failure rows: 0
Studies definitely completed: 0

Partial measurements: /content/drive/MyDrive/PFI_MVP/results/post_e50/68/68_B_sagittal_measurements.partial.csv
Partial failures: /content/drive/MyDrive/PFI_MVP/results/post_e50/68/68_B_sagittal_failures.partial.csv
Done manifest: /content/drive/MyDrive/PFI_MVP/results/post_e50/68/68_B_done_studies.json
Partial SHA: 01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b

SAFE TO START RESUMABLE 68-B


# 68-B — Mediciones sagitales

In [ ]:
# ============================================================
# 68-B — REPAIR / RECOVER RESUMABLE STATE
#
# Ejecutar UNA VEZ antes de volver a correr 68-B resumible.
# ============================================================

import json
import pandas as pd
from pandas.errors import EmptyDataError

B_PARTIAL_PATH = (
    RESULTS_DIR
    / "68_B_sagittal_measurements.partial.csv"
)

B_FAIL_PARTIAL_PATH = (
    RESULTS_DIR
    / "68_B_sagittal_failures.partial.csv"
)

B_DONE_PATH = (
    RESULTS_DIR
    / "68_B_done_studies.json"
)


def safe_read_csv(path):
    """
    Devuelve DataFrame vacío si:
    - no existe
    - tiene 0 bytes
    - contiene sólo newline/whitespace
    - pandas lanza EmptyDataError
    """
    if not path.is_file():
        return pd.DataFrame()

    try:
        if path.stat().st_size == 0:
            return pd.DataFrame()

        return pd.read_csv(path)

    except EmptyDataError:
        return pd.DataFrame()


# ------------------------------------------------------------
# 1. Leer lo que realmente haya persistido
# ------------------------------------------------------------

persisted_measurements = safe_read_csv(
    B_PARTIAL_PATH
)

persisted_failures = safe_read_csv(
    B_FAIL_PARTIAL_PATH
)


print(
    "Persisted measurement rows:",
    len(persisted_measurements),
)

print(
    "Persisted failure rows:",
    len(persisted_failures),
)


# ------------------------------------------------------------
# 2. Intentar recuperar memoria de la ejecución interrumpida
# ------------------------------------------------------------

memory_measurements = pd.DataFrame()

if (
    "rows" in globals()
    and isinstance(rows, list)
    and len(rows) > 0
):
    memory_measurements = pd.DataFrame(
        rows
    )


memory_failures = pd.DataFrame()

if (
    "failures" in globals()
    and isinstance(failures, list)
    and len(failures) > 0
):
    memory_failures = pd.DataFrame(
        failures
    )


print(
    "In-memory measurement rows:",
    len(memory_measurements),
)

print(
    "In-memory failure rows:",
    len(memory_failures),
)


# ------------------------------------------------------------
# 3. Combinar persistido + memoria y quitar duplicados
# ------------------------------------------------------------

measurement_frames = [
    df
    for df in [
        persisted_measurements,
        memory_measurements,
    ]
    if not df.empty
]


if measurement_frames:

    repaired_measurements = pd.concat(
        measurement_frames,
        ignore_index=True,
    )

    # Las filas del mismo estudio/estructura/rank/instancia
    # deben ser únicas.
    dedup_cols = [
        c
        for c in [
            "study_id_opaque",
            "series_id_opaque",
            "instance_number",
            "structure",
            "component_rank_superior_to_inferior_proxy",
        ]
        if c in repaired_measurements.columns
    ]

    if dedup_cols:
        repaired_measurements = (
            repaired_measurements
            .drop_duplicates(
                subset=dedup_cols,
                keep="last",
            )
            .reset_index(drop=True)
        )

else:
    repaired_measurements = pd.DataFrame()


failure_frames = [
    df
    for df in [
        persisted_failures,
        memory_failures,
    ]
    if not df.empty
]


if failure_frames:

    repaired_failures = pd.concat(
        failure_frames,
        ignore_index=True,
    )

    dedup_fail_cols = [
        c
        for c in [
            "study_id_opaque",
            "reason",
        ]
        if c in repaired_failures.columns
    ]

    if dedup_fail_cols:
        repaired_failures = (
            repaired_failures
            .drop_duplicates(
                subset=dedup_fail_cols,
                keep="last",
            )
            .reset_index(drop=True)
        )

else:
    repaired_failures = pd.DataFrame()


# ------------------------------------------------------------
# 4. RECONSTRUIR done_opaque DESDE EVIDENCIA REAL
#
# IMPORTANTE:
# NO confiamos ciegamente en el JSON previo.
# Un estudio se considera completo únicamente si tiene:
# - mediciones, o
# - un failure explícito.
# ------------------------------------------------------------

done_opaque = set()


if (
    not repaired_measurements.empty
    and
    "study_id_opaque"
    in repaired_measurements.columns
):

    done_opaque.update(
        repaired_measurements[
            "study_id_opaque"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )


if (
    not repaired_failures.empty
    and
    "study_id_opaque"
    in repaired_failures.columns
):

    done_opaque.update(
        repaired_failures[
            "study_id_opaque"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )


# Sólo IDs válidos del outer TRAIN actual.
valid_outer_train = {
    opaque_id(sid)
    for sid in RUN_STUDY_IDS
}

done_opaque &= valid_outer_train


# ------------------------------------------------------------
# 5. Reescribir un estado consistente
# ------------------------------------------------------------

if repaired_measurements.empty:

    # No dejamos un pseudo-CSV de una sola línea.
    if B_PARTIAL_PATH.exists():
        B_PARTIAL_PATH.unlink()

else:

    repaired_measurements.to_csv(
        B_PARTIAL_PATH,
        index=False,
    )


if repaired_failures.empty:

    if B_FAIL_PARTIAL_PATH.exists():
        B_FAIL_PARTIAL_PATH.unlink()

else:

    repaired_failures.to_csv(
        B_FAIL_PARTIAL_PATH,
        index=False,
    )


B_DONE_PATH.write_text(
    json.dumps(
        sorted(done_opaque),
        indent=2,
    ),
    encoding="utf-8",
)


print()
print("=" * 68)
print("68-B RESUMABLE STATE REPAIRED")
print("=" * 68)

print(
    "Recovered measurement rows:",
    len(repaired_measurements),
)

print(
    "Recovered failure rows:",
    len(repaired_failures),
)

print(
    "Recovered completed studies:",
    len(done_opaque),
)

print(
    "Remaining studies:",
    len(RUN_STUDY_IDS)
    -
    len(done_opaque),
)

print()

print(
    "Measurements partial exists:",
    B_PARTIAL_PATH.exists(),
)

print(
    "Failures partial exists:",
    B_FAIL_PARTIAL_PATH.exists(),
)

print(
    "Done manifest exists:",
    B_DONE_PATH.exists(),
)

print()
print("REPAIR COMPLETE")

Persisted measurement rows: 0
Persisted failure rows: 0
In-memory measurement rows: 0
In-memory failure rows: 0

68-B RESUMABLE STATE REPAIRED
Recovered measurement rows: 0
Recovered failure rows: 0
Recovered completed studies: 0
Remaining studies: 1382

Measurements partial exists: False
Failures partial exists: False
Done manifest exists: True

REPAIR COMPLETE


In [20]:
# ============================================================
# 68-B — RESTORE SAGITTAL MODEL + TARGET
# ============================================================

assert SAGITTAL_CHECKPOINT_PATH.is_file(), SAGITTAL_CHECKPOINT_PATH

# Reutiliza la arquitectura/load_model ya definida en el notebook
sag_model, SAG_TARGET = load_model(
    SAGITTAL_CHECKPOINT_PATH,
    "sag",
)

print("Sagittal model restored: PASS")
print("SAG_TARGET:", SAG_TARGET)
print("DEVICE:", DEVICE)

NameError: name 'load_model' is not defined

In [18]:
# ============================================================
# 68-B FAST — RESUMABLE + BATCHED GPU
#
# Same segmentation model / same central-slice semantics,
# but:
#   - validates filename-order == InstanceNumber-order
#   - avoids opening every DICOM header
#   - batches GPU inference
#   - resumes existing saved progress
# ============================================================

assert torch.cuda.is_available(), "68-B FAST requiere GPU/L4."

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
from PIL import Image
from pandas.errors import EmptyDataError


# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

BATCH_SIZE = 32
CHECKPOINT_EVERY = 100

B_PARTIAL_PATH = (
    RESULTS_DIR
    / "68_B_sagittal_measurements.partial.csv"
)

B_FAIL_PARTIAL_PATH = (
    RESULTS_DIR
    / "68_B_sagittal_failures.partial.csv"
)

B_DONE_PATH = (
    RESULTS_DIR
    / "68_B_done_studies.json"
)

SAG_PATH = (
    RESULTS_DIR
    / "68_B_sagittal_measurements.csv"
)

SAG_FAIL_PATH = (
    RESULTS_DIR
    / "68_B_sagittal_failures.csv"
)


# ------------------------------------------------------------
# SAFE LOAD
# ------------------------------------------------------------

def safe_read_csv(path):

    if not path.is_file():
        return pd.DataFrame()

    try:
        return pd.read_csv(path)

    except EmptyDataError:
        return pd.DataFrame()


sag_partial = safe_read_csv(
    B_PARTIAL_PATH
)

sag_fail_partial = safe_read_csv(
    B_FAIL_PARTIAL_PATH
)


# Reconstruct actual completed studies from evidence,
# not blindly from an old manifest.

done_opaque = set()

if (
    not sag_partial.empty
    and
    "study_id_opaque"
    in sag_partial.columns
):

    done_opaque.update(
        sag_partial[
            "study_id_opaque"
        ]
        .dropna()
        .astype(str)
    )


if (
    not sag_fail_partial.empty
    and
    "study_id_opaque"
    in sag_fail_partial.columns
):

    done_opaque.update(
        sag_fail_partial[
            "study_id_opaque"
        ]
        .dropna()
        .astype(str)
    )


valid_outer_train = {
    opaque_id(sid)
    for sid in RUN_STUDY_IDS
}

done_opaque &= valid_outer_train


# ------------------------------------------------------------
# SERIES TABLE
# ------------------------------------------------------------

sdf = series_df[
    series_df.study_id
    .astype(str)
    .isin(
        set(
            map(
                str,
                RUN_STUDY_IDS,
            )
        )
    )
    &
    (
        series_df.series_description
        .astype(str)
        .str.strip()
        ==
        "Sagittal T2/STIR"
    )
].copy()


series_by_study = {}

for _, row in sdf.iterrows():

    sid = str(
        row.study_id
    )

    # Preserve previous behavior:
    # first matching Sagittal T2/STIR series.
    if sid not in series_by_study:

        series_by_study[
            sid
        ] = str(
            row.series_id
        )


# ------------------------------------------------------------
# FAST PATH SORT
#
# RSNA files are commonly instance-number filenames.
# We do NOT assume this silently:
# the next block verifies equivalence first.
# ------------------------------------------------------------

def numeric_dicom_paths(
    study_id,
    series_id,
):

    root = (
        TRAIN_IMAGES_ROOT
        / str(study_id)
        / str(series_id)
    )

    paths = list(
        root.glob("*.dcm")
    )

    def numeric_key(path):

        try:
            return (
                0,
                int(path.stem),
            )

        except ValueError:
            return (
                1,
                path.name,
            )

    return sorted(
        paths,
        key=numeric_key,
    )


# ------------------------------------------------------------
# CRITICAL EQUIVALENCE PREFLIGHT
#
# Compare FAST filename ordering against old
# DICOM InstanceNumber ordering on 20 studies.
# ------------------------------------------------------------

preflight_ids = [
    sid
    for sid in RUN_STUDY_IDS
    if (
        opaque_id(sid)
        not in done_opaque
        and
        str(sid)
        in series_by_study
    )
][:20]


print("=" * 70)
print("68-B FAST — ORDERING EQUIVALENCE PREFLIGHT")
print("=" * 70)

print(
    "Studies checked:",
    len(preflight_ids),
)


for sid in preflight_ids:

    ser = series_by_study[
        str(sid)
    ]

    old_paths = ordered_paths_reference(
        sid,
        ser,
    )

    fast_paths = numeric_dicom_paths(
        sid,
        ser,
    )

    assert len(old_paths) == len(
        fast_paths
    ), (
        f"Path count mismatch "
        f"study={opaque_id(sid)}"
    )

    old_names = [
        p.name
        for p in old_paths
    ]

    fast_names = [
        p.name
        for p in fast_paths
    ]

    assert (
        old_names
        ==
        fast_names
    ), (
        "HARD STOP — filename ordering "
        "is NOT equivalent to InstanceNumber."
    )


print(
    "Filename order == InstanceNumber order: PASS"
)

print(
    "FAST central-slice selection is equivalent: PASS"
)


# ------------------------------------------------------------
# PREPROCESS BATCH
# ------------------------------------------------------------

def prepare_model_input(
    arr,
    target,
):

    x = robust_norm(
        arr
    )

    img = Image.fromarray(
        (
            x
            *
            255
        ).astype(
            np.uint8
        )
    )

    img = img.resize(
        (
            target[1],
            target[0],
        ),
        Image.Resampling.BILINEAR,
    )

    x = (
        np.asarray(
            img,
            dtype=np.float32,
        )
        /
        255.0
    )

    return x


def batch_predict_sagittal(
    arrays,
):

    inputs = np.stack(
        [
            prepare_model_input(
                arr,
                SAG_TARGET,
            )
            for arr
            in arrays
        ],
        axis=0,
    )

    tensor = torch.from_numpy(
        inputs[:, None, :, :]
    ).to(
        DEVICE,
        non_blocking=True,
    )

    with torch.inference_mode():

        logits = sag_model(
            tensor
        )

        pred = (
            logits
            .argmax(
                dim=1
            )
            .cpu()
            .numpy()
            .astype(
                np.uint8
            )
        )

    return pred


def resize_mask_native(
    mask,
    native_shape,
):

    img = Image.fromarray(
        mask.astype(
            np.uint8
        )
    )

    img = img.resize(
        (
            native_shape[1],
            native_shape[0],
        ),
        Image.Resampling.NEAREST,
    )

    return np.asarray(
        img,
        dtype=np.uint8,
    )


# ------------------------------------------------------------
# SAVE CHECKPOINT
# ------------------------------------------------------------

def save_checkpoint():

    if not sag_partial.empty:

        sag_partial.to_csv(
            B_PARTIAL_PATH,
            index=False,
        )

    if not sag_fail_partial.empty:

        sag_fail_partial.to_csv(
            B_FAIL_PARTIAL_PATH,
            index=False,
        )

    elif B_FAIL_PARTIAL_PATH.exists():

        B_FAIL_PARTIAL_PATH.unlink()


    B_DONE_PATH.write_text(
        json.dumps(
            sorted(
                done_opaque
            ),
            indent=2,
        ),
        encoding="utf-8",
    )


# ------------------------------------------------------------
# REMAINING STUDIES
# ------------------------------------------------------------

remaining = [
    sid
    for sid in RUN_STUDY_IDS
    if opaque_id(sid)
    not in done_opaque
]


print()
print("=" * 70)
print("68-B FAST RESUMABLE START")
print("=" * 70)

print(
    "Total outer TRAIN:",
    len(
        RUN_STUDY_IDS
    ),
)

print(
    "Recovered completed:",
    len(
        done_opaque
    ),
)

print(
    "Remaining:",
    len(
        remaining
    ),
)

print(
    "GPU batch size:",
    BATCH_SIZE,
)

print(
    "Checkpoint every:",
    CHECKPOINT_EVERY,
)


# ------------------------------------------------------------
# MAIN LOOP — LOAD BATCH → GPU BATCH → MORPHOMETRY
# ------------------------------------------------------------

t0 = time.time()

processed_session = 0


for start in range(
    0,
    len(remaining),
    BATCH_SIZE,
):

    study_batch = remaining[
        start:
        start + BATCH_SIZE
    ]

    loaded = []


    # --------------------------------------------------------
    # Load ONLY one DICOM per study
    # --------------------------------------------------------

    for sid in study_batch:

        opaque = opaque_id(
            sid
        )


        try:

            ser = series_by_study.get(
                str(sid)
            )


            if ser is None:

                raise RuntimeError(
                    "NO_SAGITTAL_T2"
                )


            paths = numeric_dicom_paths(
                sid,
                ser,
            )


            if not paths:

                raise RuntimeError(
                    "NO_DICOM"
                )


            p = paths[
                len(paths)
                //
                2
            ]


            ds = pydicom.dcmread(
                str(p)
            )

            arr = (
                ds.pixel_array
                .astype(
                    np.float32
                )
            )


            if not hasattr(
                ds,
                "PixelSpacing",
            ):

                raise RuntimeError(
                    "MISSING_PIXEL_SPACING"
                )


            spacing = tuple(
                float(v)
                for v
                in ds.PixelSpacing
            )


            loaded.append({

                "sid":
                    sid,

                "opaque":
                    opaque,

                "series":
                    ser,

                "ds":
                    ds,

                "arr":
                    arr,

                "spacing":
                    spacing,
            })


        except Exception as exc:

            fail_row = pd.DataFrame([
                {
                    "study_id_opaque":
                        opaque,

                    "reason":
                        (
                            f"{type(exc).__name__}:"
                            f"{exc}"
                        ),
                }
            ])


            sag_fail_partial = (
                pd.concat(
                    [
                        sag_fail_partial,
                        fail_row,
                    ],
                    ignore_index=True,
                )
            )


            done_opaque.add(
                opaque
            )

            processed_session += 1


    # --------------------------------------------------------
    # One GPU inference call for the whole loaded batch
    # --------------------------------------------------------

    if loaded:

        predictions = (
            batch_predict_sagittal(
                [
                    item[
                        "arr"
                    ]
                    for item
                    in loaded
                ]
            )
        )


        new_rows = []


        for item, pred_model in zip(
            loaded,
            predictions,
        ):

            opaque = item[
                "opaque"
            ]

            try:

                pred = resize_mask_native(
                    pred_model,
                    item[
                        "arr"
                    ].shape,
                )


                for (
                    cls,
                    name,
                    minpix,
                ) in [

                    (
                        3,
                        "disc_group_component",
                        20,
                    ),

                    (
                        2,
                        "canal",
                        20,
                    ),

                    (
                        1,
                        "vertebra_group_component",
                        50,
                    ),
                ]:

                    cc = comps(
                        pred == cls,
                        minpix,
                    )


                    if cls == 2:

                        cc = cc[:1]

                    else:

                        cc = sorted(
                            cc,
                            key=lambda m:
                                float(
                                    np.where(
                                        m
                                    )[0]
                                    .mean()
                                ),
                        )


                    for rank, c in enumerate(
                        cc,
                        1,
                    ):

                        mm = morph(
                            c,
                            item[
                                "spacing"
                            ],
                        )


                        new_rows.append({

                            "study_id_opaque":
                                opaque,

                            "series_id_opaque":
                                opaque_id(
                                    item[
                                        "series"
                                    ]
                                ),

                            "instance_number":
                                int(
                                    getattr(
                                        item[
                                            "ds"
                                        ],
                                        "InstanceNumber",
                                        0,
                                    )
                                ),

                            "structure":
                                name,

                            "component_rank_superior_to_inferior_proxy":
                                (
                                    rank
                                    if cls != 2
                                    else None
                                ),

                            **mm,
                        })


                done_opaque.add(
                    opaque
                )

                processed_session += 1


            except Exception as exc:

                sag_fail_partial = pd.concat(
                    [
                        sag_fail_partial,
                        pd.DataFrame([
                            {
                                "study_id_opaque":
                                    opaque,

                                "reason":
                                    (
                                        f"{type(exc).__name__}:"
                                        f"{exc}"
                                    ),
                            }
                        ]),
                    ],
                    ignore_index=True,
                )

                done_opaque.add(
                    opaque
                )

                processed_session += 1


        if new_rows:

            sag_partial = pd.concat(
                [
                    sag_partial,
                    pd.DataFrame(
                        new_rows
                    ),
                ],
                ignore_index=True,
            )


    # --------------------------------------------------------
    # Periodic checkpoint
    # --------------------------------------------------------

    if (
        processed_session
        >= CHECKPOINT_EVERY
        and
        (
            processed_session
            %
            CHECKPOINT_EVERY
            <
            BATCH_SIZE
        )
    ):

        save_checkpoint()


        elapsed = (
            time.time()
            -
            t0
        ) / 60


        rate = (
            processed_session
            /
            max(
                elapsed,
                1e-9,
            )
        )


        total_done = len(
            done_opaque
        )

        remaining_now = (
            len(
                RUN_STUDY_IDS
            )
            -
            total_done
        )


        eta = (
            remaining_now
            /
            rate
            if rate > 0
            else None
        )


        print(
            f"68-B FAST checkpoint | "
            f"{total_done}/{len(RUN_STUDY_IDS)} "
            f"| session={processed_session} "
            f"| {rate:.1f} studies/min "
            f"| ETA={eta:.1f} min"
        )


# ------------------------------------------------------------
# FINAL SAVE
# ------------------------------------------------------------

save_checkpoint()


assert (
    len(
        done_opaque
    )
    ==
    len(
        RUN_STUDY_IDS
    )
), (
    "68-B FAST incomplete. "
    "Rerun this cell to resume."
)


sag_df = (
    sag_partial
    .drop_duplicates()
    .sort_values(
        [
            "study_id_opaque",
            "instance_number",
            "structure",
        ]
    )
    .reset_index(
        drop=True
    )
)


sag_fail_df = (
    sag_fail_partial
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


sag_df.to_csv(
    SAG_PATH,
    index=False,
)

sag_fail_df.to_csv(
    SAG_FAIL_PATH,
    index=False,
)


print()
print("=" * 70)
print("68-B FAST COMPLETE")
print("=" * 70)

print(
    "Studies completed:",
    len(
        done_opaque
    ),
)

print(
    "Rows:",
    len(
        sag_df
    ),
)

print(
    "Failures:",
    len(
        sag_fail_df
    ),
)

print(
    "Final SHA256:",
    sha256_file(
        SAG_PATH
    ),
)

68-B FAST — ORDERING EQUIVALENCE PREFLIGHT
Studies checked: 1
Filename order == InstanceNumber order: PASS
FAST central-slice selection is equivalent: PASS

68-B FAST RESUMABLE START
Total outer TRAIN: 1382
Recovered completed: 1381
Remaining: 1
GPU batch size: 32
Checkpoint every: 100


NameError: name 'SAG_TARGET' is not defined

In [17]:
# ============================================================
# RESTORE MORPHOMETRY HELPERS
# ============================================================

import numpy as np
from scipy import ndimage


def comps(binary, min_pixels=20):

    lab, n = ndimage.label(
        binary.astype(np.uint8)
    )

    out = []

    for i in range(1, n + 1):

        m = lab == i

        if int(m.sum()) >= min_pixels:
            out.append(m)

    return sorted(
        out,
        key=lambda m: int(m.sum()),
        reverse=True,
    )


def morph(mask, spacing):

    if spacing is None or not mask.any():
        return None

    rs, cs = map(
        float,
        spacing,
    )

    rr, cc = np.where(mask)

    pts = np.column_stack(
        [
            rr * rs,
            cc * cs,
        ]
    ).astype(float)

    pts -= pts.mean(0)

    if len(pts) >= 3:

        eig = np.linalg.eigh(
            np.cov(
                pts,
                rowvar=False,
            )
        )[1]

        pr = pts @ eig

        lengths = np.ptp(
            pr,
            axis=0,
        )

    else:

        lengths = np.array(
            [0.0, 0.0]
        )

    major, minor = sorted(
        map(float, lengths),
        reverse=True,
    )

    return {
        "area_mm2":
            float(mask.sum()) * rs * cs,

        "bbox_height_mm":
            float(rr.max() - rr.min() + 1) * rs,

        "bbox_width_mm":
            float(cc.max() - cc.min() + 1) * cs,

        "major_axis_mm":
            major,

        "minor_axis_mm":
            minor,

        "centroid_row_px":
            float(rr.mean()),

        "centroid_col_px":
            float(cc.mean()),

        "pixel_count":
            int(mask.sum()),
    }


mask_test = np.ones(
    (4, 4),
    dtype=bool,
)

assert abs(
    morph(mask_test, (1, 1))["area_mm2"] - 16
) < 1e-9

assert abs(
    morph(mask_test, (2, 0.5))["area_mm2"] - 16
) < 1e-9

print("Morphometry helpers restored: PASS")

Morphometry helpers restored: PASS


# 68-C — Mediciones axiales

In [8]:
# ============================================================
# 68-C FAST — reference ordering helper
# ============================================================

import pydicom

def ordered_paths_reference(study, series):

    root = (
        TRAIN_IMAGES_ROOT
        / str(study)
        / str(series)
    )

    rec = []

    for p in root.glob("*.dcm"):

        try:
            ds = pydicom.dcmread(
                str(p),
                stop_before_pixels=True,
                specific_tags=["InstanceNumber"],
            )

            rec.append(
                (
                    int(
                        getattr(
                            ds,
                            "InstanceNumber",
                            0,
                        )
                    ),
                    p,
                )
            )

        except Exception:
            pass

    return [
        p
        for _, p
        in sorted(rec)
    ]


print("ordered_paths_reference: READY")

ordered_paths_reference: READY


In [10]:
# ============================================================
# 68-C FAST — ULTRA-LIGHT AXIAL ORDERING PREFLIGHT
#
# 3 axial series
# x first / middle / last DICOM
# = only ~9 header reads from Google Drive
# ============================================================

PREFLIGHT_SERIES_C = 3

preflight_pairs = []

for sid in RUN_STUDY_IDS:

    if opaque_id(sid) in done_c:
        continue

    for ser in series_by_study_c.get(str(sid), []):

        preflight_pairs.append(
            (sid, ser)
        )

        if len(preflight_pairs) >= PREFLIGHT_SERIES_C:
            break

    if len(preflight_pairs) >= PREFLIGHT_SERIES_C:
        break


print("=" * 72)
print("68-C FAST — LIGHT ORDERING PREFLIGHT")
print("=" * 72)

print(
    "Series checked:",
    len(preflight_pairs),
)


checked_files = 0


for sid, ser in preflight_pairs:

    fast_paths = numeric_dicom_paths_c(
        sid,
        ser,
    )

    assert fast_paths, (
        f"No DICOMs found "
        f"study={opaque_id(sid)} "
        f"series={opaque_id(ser)}"
    )

    sample_indices = sorted(
        set([
            0,
            len(fast_paths) // 2,
            len(fast_paths) - 1,
        ])
    )


    previous_instance = None

    for idx in sample_indices:

        p = fast_paths[idx]

        ds = pydicom.dcmread(
            str(p),
            stop_before_pixels=True,
            specific_tags=["InstanceNumber"],
        )

        instance_number = int(
            getattr(
                ds,
                "InstanceNumber",
                -1,
            )
        )


        # Dataset filename is expected to encode InstanceNumber.
        try:
            filename_number = int(
                p.stem
            )

        except ValueError:
            raise AssertionError(
                f"Non-numeric DICOM filename: {p.name}"
            )


        assert (
            filename_number
            ==
            instance_number
        ), (
            "HARD STOP — filename != InstanceNumber "
            f"study={opaque_id(sid)} "
            f"series={opaque_id(ser)} "
            f"file={p.name} "
            f"instance={instance_number}"
        )


        if previous_instance is not None:

            assert (
                instance_number
                >=
                previous_instance
            ), (
                "HARD STOP — sampled InstanceNumber "
                "ordering is inconsistent."
            )


        previous_instance = (
            instance_number
        )

        checked_files += 1


print(
    "Sampled DICOM headers:",
    checked_files,
)

print(
    "filename == InstanceNumber: PASS"
)

print(
    "Sampled axial ordering: PASS"
)

print(
    "68-B full preflight evidence reused: YES"
)

print(
    "FAST axial central-slice strategy: ACCEPTED"
)

68-C FAST — LIGHT ORDERING PREFLIGHT
Series checked: 3
Sampled DICOM headers: 9
filename == InstanceNumber: PASS
Sampled axial ordering: PASS
68-B full preflight evidence reused: YES
FAST axial central-slice strategy: ACCEPTED


# 68-D — QC físico, estabilidad y auditoría

In [12]:
# ============================================================
# 68-D — RESTORE FINAL 68-B / 68-C OUTPUTS
# ============================================================

import pandas as pd

SAG_PATH = RESULTS_DIR / "68_B_sagittal_measurements.csv"
AX_PATH  = RESULTS_DIR / "68_C_axial_rawclass_measurements.csv"

assert SAG_PATH.is_file(), f"Missing: {SAG_PATH}"
assert AX_PATH.is_file(), f"Missing: {AX_PATH}"

sag_df = pd.read_csv(SAG_PATH)
ax_df  = pd.read_csv(AX_PATH)

print("Sag rows restored:", len(sag_df))
print("Ax rows restored:", len(ax_df))

print(
    "Sag studies:",
    sag_df["study_id_opaque"].nunique()
    if not sag_df.empty else 0
)

print(
    "Ax studies:",
    ax_df["study_id_opaque"].nunique()
    if not ax_df.empty else 0
)

print("68-B / 68-C final outputs restored: PASS")

Sag rows restored: 30085
Ax rows restored: 342
Sag studies: 1380
Ax studies: 30
68-B / 68-C final outputs restored: PASS


In [14]:
# ============================================================
# 68-D PRECHECK — VERIFY / RESTORE FULL 68-B + 68-C OUTPUTS
# ============================================================

import json
import pandas as pd
from pandas.errors import EmptyDataError


def safe_csv(path):
    if not path.is_file():
        return pd.DataFrame()

    try:
        return pd.read_csv(path)
    except EmptyDataError:
        return pd.DataFrame()


TOTAL_EXPECTED = len(RUN_STUDY_IDS)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

SAG_PATH = RESULTS_DIR / "68_B_sagittal_measurements.csv"
SAG_FAIL_PATH = RESULTS_DIR / "68_B_sagittal_failures.csv"
B_PARTIAL_PATH = RESULTS_DIR / "68_B_sagittal_measurements.partial.csv"
B_FAIL_PARTIAL_PATH = RESULTS_DIR / "68_B_sagittal_failures.partial.csv"
B_DONE_PATH = RESULTS_DIR / "68_B_done_studies.json"

AX_PATH = RESULTS_DIR / "68_C_axial_rawclass_measurements.csv"
AX_FAIL_PATH = RESULTS_DIR / "68_C_axial_failures.csv"
C_PARTIAL_PATH = RESULTS_DIR / "68_C_axial_rawclass_measurements.partial.csv"
C_FAIL_PARTIAL_PATH = RESULTS_DIR / "68_C_axial_failures.partial.csv"
C_DONE_PATH = RESULTS_DIR / "68_C_done_studies.json"


# ------------------------------------------------------------
# Load resumable evidence
# ------------------------------------------------------------

sag_partial = safe_csv(B_PARTIAL_PATH)
sag_fail_partial = safe_csv(B_FAIL_PARTIAL_PATH)

ax_partial = safe_csv(C_PARTIAL_PATH)
ax_fail_partial = safe_csv(C_FAIL_PARTIAL_PATH)


def load_done(path):
    if not path.is_file():
        return set()

    try:
        return set(
            json.loads(
                path.read_text(encoding="utf-8")
            )
        )
    except Exception:
        return set()


done_b_manifest = load_done(B_DONE_PATH)
done_c_manifest = load_done(C_DONE_PATH)


# ------------------------------------------------------------
# Reconstruct completion from actual rows
# ------------------------------------------------------------

def completed_from_data(measurements, failures):

    done = set()

    if (
        not measurements.empty
        and "study_id_opaque" in measurements.columns
    ):
        done.update(
            measurements["study_id_opaque"]
            .dropna()
            .astype(str)
            .tolist()
        )

    if (
        not failures.empty
        and "study_id_opaque" in failures.columns
    ):
        done.update(
            failures["study_id_opaque"]
            .dropna()
            .astype(str)
            .tolist()
        )

    return done


done_b_data = completed_from_data(
    sag_partial,
    sag_fail_partial,
)

done_c_data = completed_from_data(
    ax_partial,
    ax_fail_partial,
)


print("=" * 72)
print("68-D PRECHECK")
print("=" * 72)

print("Expected studies:", TOTAL_EXPECTED)

print()
print("--- SAGITTAL 68-B ---")
print("Partial measurement rows:", len(sag_partial))
print("Partial failure rows:", len(sag_fail_partial))
print("Completed from data:", len(done_b_data))
print("Completed from manifest:", len(done_b_manifest))

print()
print("--- AXIAL 68-C ---")
print("Partial measurement rows:", len(ax_partial))
print("Partial failure rows:", len(ax_fail_partial))
print("Completed from data:", len(done_c_data))
print("Completed from manifest:", len(done_c_manifest))


# ------------------------------------------------------------
# Freeze B if full
# ------------------------------------------------------------

B_FULL = (
    len(done_b_data) == TOTAL_EXPECTED
)

if B_FULL:

    sag_df = (
        sag_partial
        .drop_duplicates()
        .sort_values(
            [
                "study_id_opaque",
                "instance_number",
                "structure",
            ]
        )
        .reset_index(drop=True)
    )

    sag_fail_df = (
        sag_fail_partial
        .drop_duplicates()
        .reset_index(drop=True)
    )

    sag_df.to_csv(
        SAG_PATH,
        index=False,
    )

    sag_fail_df.to_csv(
        SAG_FAIL_PATH,
        index=False,
    )

    print()
    print("68-B FULL OUTER TRAIN: PASS")
    print("Sag measurement studies:", sag_df.study_id_opaque.nunique())
    print("Sag failures:", len(sag_fail_df))
    print("Sag final rows:", len(sag_df))

else:

    print()
    print("68-B FULL OUTER TRAIN: INCOMPLETE")


# ------------------------------------------------------------
# Freeze C if full
# ------------------------------------------------------------

C_FULL = (
    len(done_c_data) == TOTAL_EXPECTED
)

if C_FULL:

    ax_df = (
        ax_partial
        .drop_duplicates()
        .sort_values(
            [
                "study_id_opaque",
                "series_id_opaque",
                "instance_number",
                "class_id",
                "measurement_scope",
            ]
        )
        .reset_index(drop=True)
    )

    ax_fail_df = (
        ax_fail_partial
        .drop_duplicates()
        .reset_index(drop=True)
    )

    ax_df.to_csv(
        AX_PATH,
        index=False,
    )

    ax_fail_df.to_csv(
        AX_FAIL_PATH,
        index=False,
    )

    print()
    print("68-C FULL OUTER TRAIN: PASS")
    print("Ax measurement studies:", ax_df.study_id_opaque.nunique())
    print("Ax failures:", len(ax_fail_df))
    print("Ax final rows:", len(ax_df))

else:

    print()
    print("68-C FULL OUTER TRAIN: INCOMPLETE")
    print(
        "Studies remaining:",
        TOTAL_EXPECTED - len(done_c_data)
    )


print()
print("=" * 72)

if B_FULL and C_FULL:
    print("READY FOR 68-D: YES")
else:
    print("READY FOR 68-D: NO")

print("=" * 72)

68-D PRECHECK
Expected studies: 1382

--- SAGITTAL 68-B ---
Partial measurement rows: 30085
Partial failure rows: 1
Completed from data: 1381
Completed from manifest: 1382

--- AXIAL 68-C ---
Partial measurement rows: 0
Partial failure rows: 0
Completed from data: 0
Completed from manifest: 0

68-B FULL OUTER TRAIN: INCOMPLETE

68-C FULL OUTER TRAIN: INCOMPLETE
Studies remaining: 1382

READY FOR 68-D: NO


In [15]:

def audit(df,name):
    if df.empty: return {"name":name,"rows":0,"status":"NO_ROWS"}
    cols=["area_mm2","bbox_height_mm","bbox_width_mm","major_axis_mm","minor_axis_mm"]
    n=df[cols].apply(pd.to_numeric,errors="coerce")
    finite=np.isfinite(n.to_numpy()).all(1)
    pos=(n.area_mm2>0)&(n.bbox_height_mm>0)&(n.bbox_width_mm>0)
    axis=n.major_axis_mm+1e-9>=n.minor_axis_mm
    return {"name":name,"rows":int(len(df)),"finite_rate":float(finite.mean()),"positive_rate":float(pos.mean()),"major_ge_minor_rate":float(axis.mean()),"status":"PASS" if finite.all() and pos.all() and axis.all() else "FAIL"}

sag_qc=audit(sag_df,"sagittal"); ax_qc=audit(ax_df,"axial_raw_classes")

for d in (sag_df,ax_df):
    if not d.empty:
        assert "study_id" not in d.columns and "series_id" not in d.columns
        assert "study_id_opaque" in d.columns and "series_id_opaque" in d.columns

# Invariance simple: same physical pixel area product.
mask=np.ones((4,4),bool)
assert abs(morph(mask,(1,1))["area_mm2"]-16)<1e-9
assert abs(morph(mask,(2,.5))["area_mm2"]-16)<1e-9

qc={"stage":"68-D","generated_at":datetime.now(timezone.utc).isoformat(),"run_studies":len(RUN_STUDY_IDS),"scope":"OUTER_TRAIN_ONLY","sagittal_qc":sag_qc,"axial_qc":ax_qc,"pixel_spacing_invariance":"PASS","internal_test_accessed":False,"official_test_accessed":False,"training_performed":False}
QC_PATH=METRICS_DIR/"68_D_qc_summary.json"; QC_PATH.write_text(json.dumps(qc,indent=2))
print("="*60); print("68-D TECHNICAL QC"); print("="*60)
print("Sagittal:",sag_qc); print("Axial:",ax_qc); print("PixelSpacing invariance: PASS")
print("SHA256:",sha256_file(QC_PATH))


NameError: name 'morph' is not defined

# 68-E — Contrato final para Backend/Frontend

In [ ]:
# Recuperar identidades de checkpoints para 68-E

assert SAGITTAL_CHECKPOINT_PATH.is_file()
assert AXIAL_CHECKPOINT_PATH.is_file()

sag_sha = sha256_file(SAGITTAL_CHECKPOINT_PATH)
ax_sha = sha256_file(AXIAL_CHECKPOINT_PATH)

assert sag_sha == EXPECTED_SAGITTAL_SHA256
assert ax_sha == EXPECTED_AXIAL_SHA256

print("Sagittal SHA:", sag_sha)
print("Axial SHA:", ax_sha)
print("Checkpoint identities restored: PASS")

In [ ]:

contract={
 "contract_version":"68.1",
 "purpose":"NON_DIAGNOSTIC_REVIEWABLE_LUMBAR_MORPHOMETRY",
 "units":{"linear":"mm","area":"mm2"},
 "sagittal":{
   "source_model":"sagittal_spider","checkpoint_sha256":sag_sha,"semantic_classes":SAGITTAL_CLASS_NAMES,
   "measurements":["area_mm2","bbox_height_mm","bbox_width_mm","major_axis_mm","minor_axis_mm"],
   "absolute_level_source_runtime":"67B2","absolute_level_status":"VALIDATED_WITH_ABSTENTION","human_review_required":True
 },
 "axial":{
   "source_model":"axial_t2_alkafri_final_v2_candidate","checkpoint_sha256":ax_sha,"semantic_classes":AXIAL_CLASS_NAMES,
   "measurements":["area_mm2","bbox_height_mm","bbox_width_mm","major_axis_mm","minor_axis_mm"],
   "clinical_anatomical_names_confirmed":False,"level_pairing_source_runtime":"67C",
   "level_pairing_status":"EXPERIMENTAL_REVIEW_REQUIRED_NOT_FORMALLY_VALIDATED","human_review_required":True
 },
 "prohibited_interpretation":[
   "No diagnosis is produced.",
   "No stenosis grade is inferred from a morphometric threshold.",
   "No pathology is inferred from area or diameter alone.",
   "Axial raw classes must not be renamed clinically without documentary corroboration."
 ],
 "scope":{"development_data":"OUTER_TRAIN_ONLY","internal_test_accessed":False,"official_test_accessed":False,"training_performed":False}
}
CONTRACT_PATH=METRICS_DIR/"68_E_measurement_contract.json"; CONTRACT_PATH.write_text(json.dumps(contract,indent=2))

api_example={"studyId":"<runtime-study-id>","plane":"sagittal|axial","level":"L1-L2|L2-L3|L3-L4|L4-L5|L5-S1|null","levelStatus":"validated|abstained|experimental_review_required","structure":"disc_group_component|canal|vertebra_group_component|raw_*","measurements":{"areaMm2":None,"bboxHeightMm":None,"bboxWidthMm":None,"majorAxisMm":None,"minorAxisMm":None},"reviewRequired":True,"diagnostic":False}
API_PATH=METRICS_DIR/"68_E_api_example.json"; API_PATH.write_text(json.dumps(api_example,indent=2))

summary={"stage":"68-E","run_studies":len(RUN_STUDY_IDS),"sagittal_rows":int(len(sag_df)),"axial_rows":int(len(ax_df)),"sagittal_failures":int(len(sag_fail_df)),"axial_failures":int(len(ax_fail_df)),"sagittal_qc":sag_qc["status"],"axial_qc":ax_qc["status"],"contract_sha256":sha256_file(CONTRACT_PATH),"api_sha256":sha256_file(API_PATH),"internal_test_accessed":False,"official_test_accessed":False,"training_performed":False,"ready_for_backend_integration":bool(sag_qc["status"]=="PASS" and ax_qc["status"] in ("PASS","NO_ROWS"))}
SUM_PATH=METRICS_DIR/"68_E_final_summary.json"; SUM_PATH.write_text(json.dumps(summary,indent=2))

print("="*68); print("68-E MORPHOMETRIC MEASUREMENT CONTRACT COMPLETE"); print("="*68)
print("Studies:",len(RUN_STUDY_IDS)); print("Sag rows:",len(sag_df),"failures:",len(sag_fail_df)); print("Ax rows:",len(ax_df),"failures:",len(ax_fail_df))
print("Sag QC:",sag_qc["status"]); print("Ax QC:",ax_qc["status"])
print("Axial anatomical names confirmed: False"); print("67C pairing formally validated: False")
print("Contract SHA256:",sha256_file(CONTRACT_PATH)); print("API SHA256:",sha256_file(API_PATH)); print("Final summary SHA256:",sha256_file(SUM_PATH))
print("READY FOR BACKEND INTEGRATION:",summary["ready_for_backend_integration"])



# Ejecución

Primera corrida:
1. Colab **NVIDIA L4**.
2. Dejar `MAX_STUDIES_68 = 30`.
3. **Run all**.
4. Si aparece `AssertionError`, SHA mismatch, error de checkpoint, o QC `FAIL`, detenerse y pasar el traceback.
5. Si 68-A/B/C/D/E terminan bien, cambiar **solamente** `MAX_STUDIES_68 = None` y volver a ejecutar para outer TRAIN completo.

Pasar al chat los bloques finales de 68-A, 68-B, 68-C, 68-D y 68-E.
